# Evaluación de los adaptadores GRPO o DPO en Kaggle

Compara Qwen3-1.7B con thinking antes y después de GRPO sobre las **mismas 90 preguntas de eval con datos visibles** que usa la línea base del proyecto.

0. Elige `METODO = "grpo"` (con **resultados_grpo.zip**) o `METODO = "dpo"` (con **resultados_dpo.zip** de `dpo_kaggle.ipynb`). Si la base ya se midió en este mismo entorno, `EVALUAR_BASE = False` ahorra una corrida.
1. Agrega como Input un dataset con los archivos **codigo_grpo_kaggle.zip** y **resultados_grpo.zip**. También se admiten los mismos archivos ya descomprimidos por Kaggle.
2. Activa **GPU T4 x2** e **Internet On**. Se puede ejecutar con una sola GPU.
3. Usa **Save Version → Save & Run All**. Con MUESTRAS = 1 se mide pass@1. Con MUESTRAS = 5 se obtiene también pass@5.
4. En Output, descarga **resultados_eval_grpo_pass1.zip** o **resultados_eval_grpo_pass5.zip**. Incluye un resumen y los rollouts de cada modelo.

El notebook vuelve a correr el modelo original en el mismo entorno. La línea base histórica visible fue 58/90 en pass@1 y 80/90 en pass@5. Cada corrida mantiene el límite de 4.096 tokens de esa evaluación.

In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys
import tempfile
import zipfile

METODO = "grpo"  # "grpo" o "dpo": que adaptadores evaluar
MUESTRAS = 1
EVALUAR_BASE = True
LOTE = 8

SIMULADO = os.environ.get("GRPO_EVAL_SIMULADA") == "1"
ENTRADA = Path(os.environ.get("GRPO_EVAL_ENTRADA", "/kaggle/input"))
SALIDA = Path(os.environ.get("GRPO_EVAL_SALIDA", "/kaggle/working"))
TRABAJO = SALIDA / f"evaluacion_{METODO}_pass{MUESTRAS}"
MODELO = "Qwen/Qwen3-1.7B"
HUELLA_EVAL = "efb77852b9c86edc5c478be0de30fa11515d288e272a6cf09f624c3bd0453e5b"

assert METODO in ("grpo", "dpo"), "METODO debe ser grpo o dpo."
assert MUESTRAS in (1, 5), "MUESTRAS debe ser 1 o 5."
if not ENTRADA.is_dir():
    raise SystemExit("Agrega los dos ZIP con Add Input.")
TRABAJO.mkdir(parents=True, exist_ok=True)
TEMP = tempfile.TemporaryDirectory(prefix="eval_grpo_")
rutas = sorted(Path(carpeta) / nombre for carpeta, _, nombres in os.walk(ENTRADA, followlinks=True)
               for nombre in nombres)
archivos_zip = [ruta for ruta in rutas if ruta.suffix.lower() == ".zip"]

def encontrar(relativo):
    partes = relativo.split("/")
    for ruta in rutas:
        if list(ruta.parts[-len(partes):]) == partes:
            return ruta.parents[len(partes) - 1]
    for indice, ruta in enumerate(archivos_zip):
        try:
            with zipfile.ZipFile(ruta) as archivo:
                miembro = next((n for n in archivo.namelist() if n == relativo or n.endswith("/" + relativo)), None)
                if miembro:
                    destino = Path(TEMP.name) / str(indice)
                    archivo.extractall(destino)
                    return (destino / miembro).parents[len(partes) - 1]
        except zipfile.BadZipFile:
            continue
    raise SystemExit(f"Falta {relativo}. Revisa los ZIP agregados en Add Input.")

CODIGO = encontrar("scripts/run_pilot.py")
RESULTADOS = encontrar(f"runs/{METODO}_s2026/adapter/adapter_config.json")
ADAPTADORES = {f"{METODO}_s{s}": RESULTADOS / "runs" / f"{METODO}_s{s}" / "adapter" for s in (2026, 2027)
               if (RESULTADOS / "runs" / f"{METODO}_s{s}" / "adapter").is_dir()}
for nombre, ruta in ADAPTADORES.items():
    config = json.loads((ruta / "adapter_config.json").read_text(encoding="utf-8"))
    assert config["base_model_name_or_path"] == MODELO, f"Modelo base inesperado para {nombre}."
    assert (ruta / "adapter_model.safetensors").is_file(), f"Faltan los pesos de {nombre}."
assert "eval_final" in (CODIGO / "scripts/run_pilot.py").read_text(encoding="utf-8")
print("Código y ambos adaptadores encontrados.")

In [ ]:
if SIMULADO:
    n_gpu = 2
else:
    import torch

    n_gpu = torch.cuda.device_count()
    if n_gpu < 1:
        raise SystemExit("Activa GPU desde Settings antes de ejecutar.")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "transformers==5.24.0", "peft==0.22.0", "accelerate==1.16.0", "jinja2>=3.1.6"],
        check=True,
    )
    from huggingface_hub import snapshot_download

    snapshot_download(MODELO)
    print(f"{n_gpu} GPU disponibles.")

In [ ]:
from concurrent.futures import ThreadPoolExecutor

corridas = list(ADAPTADORES)
if EVALUAR_BASE:
    corridas.insert(0, "base")
n_workers = min(n_gpu, len(corridas))

def evaluar(nombre, gpu):
    carpeta = TRABAJO / nombre
    carpeta.mkdir(parents=True, exist_ok=True)
    orden = [
        sys.executable, "-u", "scripts/run_pilot.py", "--stage", "eval_final",
        "--models", MODELO, "--thinking", "on", "--conditions", "visible",
        "--max-new-tokens", "4096", "--batch-size", str(LOTE), "--n-samples", str(MUESTRAS),
        "--out", str(carpeta),
    ]
    if nombre in ADAPTADORES:
        orden += ["--adapter", str(ADAPTADORES[nombre]), "--adapter-name", nombre]
    entorno = dict(os.environ, PYTHONIOENCODING="utf-8")
    if SIMULADO:
        orden += ["--dry-run", "--limit", "9"]
    else:
        entorno["CUDA_VISIBLE_DEVICES"] = str(gpu)
    print(f"Inicia {nombre} en GPU {gpu}.", flush=True)
    with (carpeta / "ejecucion.log").open("w", encoding="utf-8") as log:
        with subprocess.Popen(
            orden, cwd=CODIGO, env=entorno, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding="utf-8", errors="replace",
        ) as proceso:
            for linea in proceso.stdout:
                log.write(linea)
                print(f"[{nombre}] {linea.rstrip()}", flush=True)
            return proceso.wait()

def evaluar_en_gpu(gpu):
    return {nombre: evaluar(nombre, gpu) for nombre in corridas[gpu::n_workers]}

with ThreadPoolExecutor(max_workers=n_workers) as ejecutor:
    codigos = {nombre: codigo for datos in ejecutor.map(evaluar_en_gpu, range(n_workers))
               for nombre, codigo in datos.items()}
print("Códigos de salida", codigos)

In [ ]:
from collections import defaultdict

def metricas(carpeta):
    manifest = json.loads((carpeta / "manifest.json").read_text(encoding="utf-8"))
    assert manifest["dataset"]["fingerprint"] == HUELLA_EVAL, "La huella difiere de la línea base."
    assert manifest["conditions"] == ["visible"] and manifest["n_samples"] == MUESTRAS
    por_caso = defaultdict(dict)
    for archivo in carpeta.glob("rollouts_*.jsonl"):
        for linea in archivo.read_text(encoding="utf-8").splitlines():
            fila = json.loads(linea)
            veredicto = fila["verdict"]
            if veredicto["error_type"] != "internal_error":
                por_caso[fila["instance_id"]][fila["sample"]] = veredicto["reward"] == 1
    completos = len(por_caso) == (9 if SIMULADO else 90) and all(
        set(respuestas) == set(range(MUESTRAS)) for respuestas in por_caso.values()
    )
    if not completos:
        raise ValueError(f"Evaluación incompleta: {len(por_caso)} casos con algún resultado.")
    primer = sum(respuestas[0] for respuestas in por_caso.values())
    varios = sum(any(respuestas.values()) for respuestas in por_caso.values())
    return primer, varios, len(por_caso)

titulo = f"# Evaluación simulada de {METODO.upper()}" if SIMULADO else f"# Evaluación de {METODO.upper()} con datos visibles"
lineas = [titulo, "", "| Modelo | pass@1 | pass@5 |", "|---|---:|---:|"]
for nombre in corridas:
    carpeta = TRABAJO / nombre
    try:
        if codigos[nombre] != 0:
            raise ValueError(f"El proceso salió con código {codigos[nombre]}.")
        primero, varios, n = metricas(carpeta)
        subprocess.run([sys.executable, "scripts/analyze_results.py", str(carpeta)], cwd=CODIGO, check=True)
        pass5 = f"{varios}/{n}" if MUESTRAS == 5 else "Pendiente"
        lineas.append(f"| {nombre} | {primero}/{n} | {pass5} |")
    except (ValueError, OSError, subprocess.CalledProcessError) as error:
        print(f"{nombre}: {error}")
        lineas.append(f"| {nombre} | Incompleta | Incompleta |")
if not SIMULADO:
    lineas.append("| Base histórica | 58/90 | 80/90 |")
lineas.extend(["", "La comparación principal usa los modelos corridos aquí con el mismo protocolo de evaluación."])
resumen = "\n".join(lineas) + "\n"
(TRABAJO / "resumen.md").write_text(resumen, encoding="utf-8")
print(resumen)

zip_salida = SALIDA / f"resultados_eval_{METODO}_pass{MUESTRAS}.zip"
with zipfile.ZipFile(zip_salida, "w", compression=zipfile.ZIP_DEFLATED) as archivo:
    for ruta in sorted(TRABAJO.rglob("*")):
        if ruta.is_file():
            archivo.write(ruta, ruta.relative_to(TRABAJO))
print("Disponible en Output:", zip_salida.name)